<a href="https://colab.research.google.com/github/flexfengfeng/SMU-Hackathon-Prep/blob/main/track3_fintech_weathershield.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌦️ Track 3 · FinTech / InsurTech
## WeatherShield: Climate Income Cover for Heat, Rain and Haze

**Coaching session · ~60 min**

### The sponsor's challenge
**Income Insurance** asks:
> *"How might we reimagine insurance, powered by data and AI, so that climate risk protection becomes affordable, accessible and actionable for every vulnerable household and SME in Singapore?"*

### The real-world problem
Singapore is getting hotter and wetter, and this year's haze was the longest since 2015. When it is too hot, too wet or too hazy to work:
- a **delivery rider** stops riding and loses the day's earnings (about 71,600 residents were regular platform workers in 2025),
- a **hawker** sees the lunch crowd stay home,
- a **renter** in a flood-prone block cleans up a flooded flat with no contents cover.

Work injury compensation covers injuries, not income lost by *stopping work as a precaution*. From 1 Dec 2026 four more heat-stress measures become mandatory for outdoor work, so rest breaks (and lost hours) will become more common. Traditional claims are slow: someone must assess every loss, and a haze episode hits everyone at once.

### The InsurTech angle
**Parametric insurance** pays when a published index crosses a threshold (NEA heat stress, 24-hour PSI, rain gauge), not after a loss adjuster visits. No claim forms, payout the next day. The hard parts are not the AI. They are **choosing the trigger**, **back-testing it**, **pricing it fairly**, and **explaining it in one SMS**. That is what we build. WeatherShield is designed **with Income as the licensed insurer**, not as a new insurer.

### What you will build
| Step | Technique | What it does |
|---|---|---|
| Index data | `pandas` + synthetic generator (+ optional live data.gov.sg feed) | 11+ years of daily WBGT, rainfall and PSI for 4 areas |
| Product design | Plain Python config | Segments, tiered triggers on official bands, payouts, day caps |
| Back-test | `pandas` + `matplotlib` | Payouts per year, zero-payout years, **basis risk** |
| Pricing | Unit economics in `pandas` | Pure premium, expense and risk loading, who pays |
| Explain | **Batch structured output** | One SMS per language, checked by Python |
| Say-back test | **LLM-as-judge** | Simulated customers paraphrase the SMS; a judge scores understanding |
| Point of sale | **Structured extraction + rules** | Free text ➜ insurable interest and cover size |
| Proposal | **"pandas calculates, LLM narrates"** | Back-test and pricing written up for the proposal |
| Demo | **Gradio** | Pick a segment and area, see triggers, back-test, premium and SMS |

## Step 1: Install Dependencies
- `google-genai`: Google's Gen AI SDK for Gemini (already preinstalled in Colab).
- `gradio`: the web UI for the demo.

`pandas`, `matplotlib`, `requests` and `pydantic` are already in Colab.

The first line records the versions of a few core packages Colab depends on, so the install below doesn't upgrade them and break Colab. If Colab asks you to **restart the session** after installing, click restart and continue from Step 2.

In [ ]:
# Keep Colab's own versions of these packages to avoid dependency conflicts
!pip freeze | grep -iE '^(google-auth|opentelemetry-(api|sdk))==' > /tmp/colab-constraints.txt
!pip install -q -c /tmp/colab-constraints.txt google-genai gradio==6.29.1

## Step 2: Setup Google Gemini API Key
1. Go to [Google AI Studio](https://aistudio.google.com/).
2. Sign in with your Google account and click **Get API key**.
3. In Colab, click the 🔑 **Secrets** icon on the left sidebar.
4. Create a new secret named `GEMINI_API_KEY` and paste your key.
5. Turn on the **Notebook access** toggle for that secret.

We also set the client to wait and retry when the free tier says *"429: too many requests"*. Free-tier keys only allow about 5 requests per minute, so every team will hit this during the hackathon. If a cell seems to pause for up to a minute, it is waiting out the limit, not stuck.

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

# Free-tier keys only allow a few requests per minute. These options make the SDK
# wait and retry any single request that gets "429: too many requests".
RETRY = types.HttpRetryOptions(attempts=6, initial_delay=15, max_delay=60, http_status_codes=[429, 500, 503])

# Securely load the API key from Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY, http_options=types.HttpOptions(retry_options=RETRY))
    print("✅ Gemini API Key configured successfully!")
except Exception as e:
    print("❌ Error: Please ensure you added 'GEMINI_API_KEY' to your Colab Secrets and turned on 'Notebook access'.")
    print("   Details:", e)
    raise

# One place to change the model for the whole notebook
MODEL = "gemini-3.5-flash-lite"

def ask_gemini(contents, config=None):
    """Call Gemini. Rate-limit retries are handled by the client (see RETRY above)."""
    return client.models.generate_content(model=MODEL, contents=contents, config=config)

## Step 3: Build the Index History (Heat, Rain, Haze)
A parametric product is only as good as its **index**. Income asks for indexes people already act on: **NEA heat stress (WBGT)**, **24-hour PSI** and **rain gauges**. They are independent, published, timely and verifiable by anyone.

For a real back-test you would download:
- NEA **WBGT observations** (data.gov.sg dataset `d_87884af1f85d702d4f74c6af13b4853d`). The WBGT network is only a few years old, so for longer history estimate WBGT from MSS temperature and humidity records.
- NEA **rainfall** (`d_72ef872af64a49bbf85c037e714267c0`) and **MSS historical daily records** (decades of rain gauge data).
- NEA historical **24-hour PSI** by region.

In this session we use a **synthetic** history (fictional numbers with realistic patterns) so the notebook runs offline in seconds:
- 🌡️ heat peaks in **April–May** and again around **September**, cooler in **December–January**, with a slow warming trend,
- 🌧️ wettest in the **north-east monsoon** (November–January), driest in February, with heavy-rain days slowly becoming heavier,
- 🌫️ haze episodes like **2015** (severe), **2019**, a short one in **2023**, and a long one **this year (2026)**.

`np.random.default_rng(42)` makes the "random" data the same every time, so everyone in the room sees the same results.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
dates = pd.date_range("2015-01-01", "2026-10-31", freq="D")
n, month = len(dates), dates.month.values
trend = (dates.year.values - 2015) + dates.dayofyear.values / 365   # years since 2015

# area: (heat offset °C, rain multiplier, PSI region, near a flood hotspot in our mock map, lat, lon)
AREAS = {
    "Jurong West": (0.3, 1.05, "west", False, 1.340, 103.709),
    "Woodlands":   (0.0, 0.95, "north", False, 1.436, 103.787),
    "Bedok":       (-0.2, 1.00, "east", True, 1.324, 103.927),
    "Kallang":     (0.4, 1.05, "central", True, 1.310, 103.871),
}
FLOOD_PRONE = [a for a, v in AREAS.items() if v[3]]

# Seasonality by month (Jan..Dec)
HEAT_BY_MONTH = np.array([-0.9, -0.5, 0.2, 0.8, 0.9, 0.5, 0.2, 0.2, 0.5, 0.3, -0.3, -0.8])
WET_PROB_BY_MONTH = np.array([0.55, 0.35, 0.45, 0.55, 0.50, 0.45, 0.45, 0.45, 0.45, 0.55, 0.65, 0.65])
RAIN_SCALE_BY_MONTH = np.array([21, 14, 15, 16, 16, 14, 14, 15, 15, 16, 20, 24])

# Haze is regional: one PSI series hits every area at once. (start, end, peak 24-hour PSI)
HAZE = [("2015-09-01", "2015-10-31", 290), ("2019-09-09", "2019-09-28", 215),
        ("2023-10-05", "2023-10-12", 115), ("2026-08-10", "2026-10-15", 175)]
psi_region = 45 + 10 * np.isin(month, [6, 7, 8, 9]) + rng.normal(0, 8, n)
for start, end, peak in HAZE:
    m = (dates >= start) & (dates <= end)
    psi_region[m] += (peak - 50) * np.sin(np.linspace(0, np.pi, m.sum())) ** 0.7 * rng.uniform(0.55, 1.1, m.sum())

frames = []
for area, (heat_off, rain_mult, region, flood, lat, lon) in AREAS.items():
    noise = rng.normal(0, 0.7, n)
    for i in range(1, n):                      # hot spells last a few days
        noise[i] += 0.6 * noise[i - 1]
    wet = rng.random(n) < WET_PROB_BY_MONTH[month - 1]
    rain = np.where(wet, rng.gamma(0.75, RAIN_SCALE_BY_MONTH[month - 1] * rain_mult * (1 + 0.012 * trend)), 0.0)
    wbgt = 29.8 + heat_off + HEAT_BY_MONTH[month - 1] + 0.03 * trend + noise - 0.04 * np.minimum(rain, 40)
    frames.append(pd.DataFrame({"date": dates, "area": area, "wbgt_max": wbgt.round(1), "rain_mm": rain.round(1),
                                "psi_24h": np.clip(psi_region + rng.normal(0, 5, n), 15, None).round().astype(int)}))
wx = pd.concat(frames, ignore_index=True)
wx["year"] = wx.date.dt.year

print(f"{len(wx):,} area-days, {wx.date.min():%d %b %Y} to {wx.date.max():%d %b %Y}")
full = wx[wx.year <= 2025]
full.groupby("area").agg(rain_mm_per_year=("rain_mm", lambda s: s.sum() / 11),
                         days_rain_50mm=("rain_mm", lambda s: (s >= 50).sum() / 11),
                         days_wbgt_high=("wbgt_max", lambda s: (s >= 33).sum() / 11),
                         days_psi_unhealthy=("psi_24h", lambda s: (s > 100).sum() / 11)).round(1)

### Optional: peek at the real live feed
The same three indexes are published every few minutes on data.gov.sg. This cell reads the **current** WBGT, rainfall and PSI near our four areas, just to show the feed is real. If the internet or the API is down, it says so and the rest of the notebook carries on with the synthetic history.

**Station vs street:** a rider on hot asphalt feels more heat than the WBGT station on a grassy stadium field. We handle that gap with a published, fixed **worksite adjustment** (`WBGT_ADJ`, e.g. +1.0°C for outdoor workers) added to the station reading in Step 4. It is the same for every customer, so anyone can check the trigger.

In [ ]:
import requests

LIVE = "https://api-open.data.gov.sg/v2/real-time/api/"

def nearest(lat, lon, items):
    """items: list of (lat, lon, value). Returns the value of the closest one."""
    return min(items, key=lambda x: (x[0] - lat) ** 2 + (x[1] - lon) ** 2)[2]

def fetch_live():
    live = pd.DataFrame(index=list(AREAS))
    try:
        wbgt = requests.get(LIVE + "weather?api=wbgt", timeout=10).json()["data"]["records"][0]["item"]["readings"]
        pts = [(float(r["location"]["latitude"]), float(r["location"]["longitude"]), float(r["wbgt"])) for r in wbgt]
        live["wbgt_now"] = [nearest(v[4], v[5], pts) for v in AREAS.values()]
    except Exception as e:
        print("⚠️ WBGT feed unavailable:", e)
    try:
        rain = requests.get(LIVE + "rainfall", timeout=10).json()["data"]
        loc = {s["id"]: s["location"] for s in rain["stations"]}
        pts = [(loc[r["stationId"]]["latitude"], loc[r["stationId"]]["longitude"], r["value"]) for r in rain["readings"][0]["data"]]
        live["rain_mm_last_5min"] = [nearest(v[4], v[5], pts) for v in AREAS.values()]
    except Exception as e:
        print("⚠️ Rainfall feed unavailable:", e)
    try:
        psi = requests.get(LIVE + "psi", timeout=10).json()["data"]["items"][0]["readings"]["psi_twenty_four_hourly"]
        live["psi_24h_now"] = [psi[v[2]] for v in AREAS.values()]
    except Exception as e:
        print("⚠️ PSI feed unavailable:", e)
    return live

live = fetch_live()
print("Live readings (data.gov.sg)" if len(live.columns) else "No live data. That's fine: the synthetic history is all we need.")
live

## Step 4: Design the Product as Plain Python Config
Every product rule lives in one readable config: **segments**, **triggers**, **payout per trigger day**, **annual cap**. Actuaries, compliance and judges can read it, and the back-test, pricing, SMS and demo all use these same numbers. Change a number here and everything downstream updates.

**Triggers use the official bands people already know:**
| Peril | Index | Official bands | WeatherShield pays from |
|---|---|---|---|
| 🌡️ Heat | NEA heat stress, daily max WBGT (+ worksite adjustment) | Low < 31 · Moderate 31 to < 33 · **High ≥ 33** | High |
| 🌧️ Rain | Daily rainfall at the area's rain gauge | (our tiers) < 50 mm · **Heavy 50 to < 70 mm** · **Very heavy 70 to < 100 mm** · **Extreme ≥ 100 mm** | Heavy (renters: Extreme, a flash-flood proxy) |
| 🌫️ Haze | NEA 24-hour PSI | Good/Moderate 0–100 · **Unhealthy 101–200** · **Very unhealthy 201–300** · **Hazardous > 300** | Unhealthy |

Showing the **S$0 rows** on purpose makes the no-pay level obvious, as Income asks.

**Rules that keep it affordable and fair:** one payout per day (the highest tier that day, no stacking); a cap on paid days per year; heat, rain and haze **bundled** in one policy; prices set by **area and peril only**, never by age, gender or race.

### 🌏 Lessons from schemes overseas, built in
- **India SEWA heat cover:** 2023 paid nothing because thresholds were set at *record* heat; 2024 paid out after resetting to local history. ➜ Our thresholds come from the back-test, and we count zero-payout years.
- **ZhongAn 37°C cover (China):** one temperature threshold ignored humidity and paid almost nothing in a cool summer; the regulator warned of gambling-like products. ➜ We use WBGT (heat *and* humidity), several tiers, and sell only where there is a real loss.
- **Korea KB stallholder cover:** no subscribers, because nobody would carry the premium. ➜ We decide **who pays** before launch (Step 6).
- **Indonesia Igloo × DANA rain cover:** people could buy only when rain was forecast. ➜ Annual or monthly cover with a **14-day waiting period**.

In [ ]:
HEAT_BANDS = {"Low": 0, "Moderate": 31, "High": 33}                      # NEA heat stress, WBGT °C
RAIN_BANDS = {"Below 50 mm": 0, "Heavy": 50, "Very heavy": 70, "Extreme": 100}  # daily rainfall, mm
HAZE_BANDS = {"Good/Moderate": 0, "Unhealthy": 101, "Very unhealthy": 201, "Hazardous": 301}  # 24-hour PSI

SEGMENTS = {
    "Outdoor / platform worker": {
        "who": "delivery riders, drivers, cleaners and construction workers who lose earnings when they stop work",
        "areas": list(AREAS), "wbgt_adj": 1.0, "cap_days": 20,
        "pays": {"heat": {"High": 8}, "rain": {"Heavy": 20, "Very heavy": 35, "Extreme": 35},
                 "haze": {"Unhealthy": 20, "Very unhealthy": 35, "Hazardous": 35}},
    },
    "Hawker / small shop": {
        "who": "hawkers, neighbourhood shops and home-based businesses that lose trade when customers stay home",
        "areas": list(AREAS), "wbgt_adj": 0.5, "cap_days": 10,
        "pays": {"heat": {"High": 5}, "rain": {"Heavy": 25, "Very heavy": 40, "Extreme": 40},
                 "haze": {"Unhealthy": 25, "Very unhealthy": 40, "Hazardous": 40}},
    },
    "Renter household": {
        "who": "lower-income renters and seniors in flood-prone blocks with no contents cover",
        "areas": FLOOD_PRONE, "wbgt_adj": 0.0, "cap_days": 2,
        "pays": {"heat": {}, "rain": {"Extreme": 150},
                 "haze": {"Very unhealthy": 30, "Hazardous": 30}},
    },
}
EXCLUSIONS = [
    "Pays only on the published readings for your registered area. You can lose income and get nothing.",
    "One payout per day (the highest tier), up to the yearly cap of paid days.",
    "Cover starts 14 days after you join.",
    "Does not pay for injury or medical bills (work injury compensation covers that).",
]

def band(value, bands):
    """Return the highest band whose lower limit the value reaches."""
    return [name for name, lower in bands.items() if value >= lower][-1]

def tier_table(segment):
    p = SEGMENTS[segment]["pays"]
    rows = [("🌡️ Heat (WBGT)", name, f"≥ {lo}" if lo else "< 31", p["heat"].get(name, 0)) for name, lo in HEAT_BANDS.items()]
    rows += [("🌧️ Rain (mm/day)", name, f"≥ {lo}" if lo else "< 50", p["rain"].get(name, 0)) for name, lo in RAIN_BANDS.items()]
    rows += [("🌫️ Haze (24h PSI)", name, f"≥ {lo}" if lo else "≤ 100", p["haze"].get(name, 0)) for name, lo in HAZE_BANDS.items()]
    return pd.DataFrame(rows, columns=["Peril", "Band", "Reading", "Payout per day (S$)"])

print(f"Worksite adjustment for outdoor workers: station WBGT + {SEGMENTS['Outdoor / platform worker']['wbgt_adj']}°C")
tier_table("Outdoor / platform worker")

## Step 5: Back-test the Trigger with pandas
Income's warning: *schemes whose first season paid nothing lost their customers*. So before selling, we replay 2015–2025 and ask:
1. **How often would it have paid**, and how much, each year? Are there **zero-payout years**?
2. **Basis risk:** on how many days did a customer *really lose income* but get **no payout**?

For (2) we need "loss days". Real teams would use platform earnings data or a survey. Here we simulate them with simple, honest assumptions: your street gets more or less rain than the gauge, the worst rain may fall in your peak hours, and worksite heat varies around the adjusted station reading.

In [ ]:
# Synthetic "what really happened at your location" noise, fixed per area-day
loss_rng = np.random.default_rng(7)
wx["local_rain"] = wx.rain_mm * loss_rng.lognormal(0, 0.6, len(wx))   # your street vs the gauge
wx["peak_share"] = loss_rng.beta(2, 3, len(wx))                        # share of rain in your busy hours
wx["site_heat"] = loss_rng.normal(0, 0.6, len(wx))                     # worksite vs adjusted station
wx["luck"] = loss_rng.random(len(wx))

def loss_day(d, segment):
    adj = SEGMENTS[segment]["wbgt_adj"]
    if segment == "Outdoor / platform worker":
        return (d.wbgt_max + adj + d.site_heat >= 33.3) | (d.local_rain * d.peak_share >= 22) | ((d.psi_24h > 100) & (d.luck < 0.7))
    if segment == "Hawker / small shop":
        return (d.wbgt_max + adj + d.site_heat >= 33.5) | (d.local_rain * d.peak_share >= 15) | ((d.psi_24h > 100) & (d.luck < 0.8))
    return (d.local_rain >= 90) & (d.luck < 0.5)          # renter: not every storm floods your flat

def backtest(segment, area):
    """Daily replay of one segment in one area: trigger, capped payout and loss day."""
    s = SEGMENTS[segment]
    d = wx[wx.area == area].copy()
    pay = pd.DataFrame({
        "heat": (d.wbgt_max + s["wbgt_adj"]).apply(lambda v: s["pays"]["heat"].get(band(v, HEAT_BANDS), 0)),
        "rain": d.rain_mm.apply(lambda v: s["pays"]["rain"].get(band(v, RAIN_BANDS), 0)),
        "haze": d.psi_24h.apply(lambda v: s["pays"]["haze"].get(band(v, HAZE_BANDS), 0)),
    }, index=d.index)
    d["trigger"] = pay.max(axis=1)                                      # highest tier only, no stacking
    d["peril"] = pay[["haze", "rain", "heat"]].idxmax(axis=1).where(d.trigger > 0, "")
    d["day_no"] = d.trigger.gt(0).groupby(d.year).cumsum()              # nth trigger day this year
    d["paid"] = d.trigger.where(d.day_no <= s["cap_days"], 0)
    d["loss"] = loss_day(d, segment)
    return d

def yearly(d):
    return d.groupby("year").agg(
        trigger_days=("trigger", lambda x: (x > 0).sum()), paid_days=("paid", lambda x: (x > 0).sum()),
        payout=("paid", "sum"), loss_days=("loss", "sum"),
        loss_days_unpaid=("paid", lambda x: (d.loc[x.index, "loss"] & (x == 0)).sum()))

results = {(seg, area): backtest(seg, area) for seg, s in SEGMENTS.items() for area in s["areas"]}
yearly(results[("Outdoor / platform worker", "Jurong West")])

Now a summary for every segment and area over the 11 full years (2015–2025). We also replay the **SEWA 2023 mistake**: a heat trigger set at *record* heat (the hottest station reading of 2015–2019) instead of the official High band. Count the years in which it would never have fired.

In [ ]:
def basis_risk(d):
    loss = d[d.loss]
    return pd.Series({
        "avg_payout_per_year": d.groupby("year").paid.sum().mean(),
        "zero_payout_years": (d.groupby("year").paid.sum() == 0).sum(),
        "loss_days_per_year": len(loss) / d.year.nunique(),
        "pct_loss_days_no_trigger": (loss.trigger == 0).mean() * 100,
        "pct_loss_days_cap_reached": ((loss.trigger > 0) & (loss.paid == 0)).mean() * 100,
        "pct_paid_days_without_loss": (~d[d.paid > 0].loss).mean() * 100,
    })

backtest_summary = pd.DataFrame({k: basis_risk(d[d.year <= 2025]) for k, d in results.items()}).T.round(1)
backtest_summary.index.names = ["segment", "area"]

jw = full[full.area == "Jurong West"]
RECORD_WBGT = jw[jw.year <= 2019].wbgt_max.max()
heat_check = pd.DataFrame({"days_High_ours": (jw.wbgt_max + SEGMENTS["Outdoor / platform worker"]["wbgt_adj"] >= HEAT_BANDS["High"]).groupby(jw.year).sum(),
                           f"days_record_{RECORD_WBGT}": (jw.wbgt_max >= RECORD_WBGT).groupby(jw.year).sum()})
record_zero_years = (heat_check.iloc[:, 1] == 0).sum()
print(f"Heat trigger days per year, outdoor worker in Jurong West. Record trigger: {record_zero_years} of 11 years with zero heat payouts")
display(heat_check.T)
backtest_summary

In [ ]:
import matplotlib.pyplot as plt

def backtest_chart(segment, area):
    d = results[(segment, area)]
    by_peril = d.pivot_table(index="year", columns="peril", values="paid", aggfunc="sum").reindex(columns=["heat", "rain", "haze"]).fillna(0)
    y = yearly(d)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
    by_peril.plot(kind="bar", stacked=True, ax=ax1, color=["#e4572e", "#2e86ab", "#8d8d8d"])
    ax1.set_title(f"Payouts per year: {segment}, {area}", fontsize=10); ax1.set_ylabel("S$ per policy"); ax1.set_xlabel("")
    for i, (yr, v) in enumerate(y.payout.items()):
        ax1.text(i, v + 2, "S$0!" if v == 0 else f"{v:.0f}", ha="center", fontsize=7)
    pd.DataFrame({"paid": y.loss_days - y.loss_days_unpaid, "unpaid": y.loss_days_unpaid}).plot(
        kind="bar", stacked=True, ax=ax2, color=["#3bb273", "#f2a541"])
    ax2.set_title("Loss days: paid vs not paid (basis risk)", fontsize=10); ax2.set_ylabel("days"); ax2.set_xlabel("")
    for ax in (ax1, ax2):
        ax.tick_params(axis="x", labelrotation=0, labelsize=7)
        ax.set_xticks(range(len(y)), [str(v) if v < 2026 else "2026\nYTD" for v in y.index])
    plt.tight_layout()
    return fig

fig = backtest_chart("Outdoor / platform worker", "Jurong West")
plt.show()

### 🤔 Coaching discussion (5 min)
- Find **2015** and **2026** in the chart. Haze hits **every** customer in **every** area on the same days. What does that mean for Income's cash flow, and why does Income need **reinsurance** for haze?
- Look at `pct_loss_days_no_trigger`: how many real loss days pay nothing? Is that acceptable? Which change helps most: a lower rain threshold, more rain gauges per area, or an hourly trigger? What does each do to the price? (Jakarta's flood card failed because residents didn't see one city gauge as *their* risk.)
- The "record heat" trigger has many zero-payout years, and the **renter** cover pays in only some years. How would you explain a S$0 year to someone who paid premiums all year? What could keep them renewing (Guatemala's MiCRO bundled cover with micro-loans and kept 70–80% renewal)?
- Is a **+1.0°C** worksite adjustment fair to a rider in Jurong West and a cleaner under a shady void deck? Who should decide the number, and how would you check it?

## Step 6: Pricing and Unit Economics
Simple, explainable pricing (MAS FEAT asks that AI and data-driven pricing can be explained):

1. **Pure premium** = average yearly payout in the back-test (what claims cost).
2. **Risk loading** = 25% of the year-to-year standard deviation of payouts. Volatile years (the 2015 haze) cost more to reinsure, so a volatile product pays more.
3. **Gross premium** = (pure + risk loading) ÷ (1 − 20% expense ratio). Income asks that expenses stay within about 20% of premium.

Inputs are **area and peril only**. There is no age, gender, race or income in the pricing table, so the model cannot price out the groups who most need cover.

In [ ]:
EXPENSE_RATIO, RISK_LOAD = 0.20, 0.25

rows = []
for (seg, area), d in results.items():
    annual = d[d.year <= 2025].groupby("year").paid.sum()
    pure, risk = annual.mean(), RISK_LOAD * annual.std()
    gross = (pure + risk) / (1 - EXPENSE_RATIO)
    rows.append({"segment": seg, "area": area, "pure_premium": pure, "risk_loading": risk,
                 "gross_per_year": gross, "per_month": gross / 12, "per_week": gross / 52,
                 "loss_ratio_pct": pure / gross * 100,
                 "haze_share_pct": d[(d.year <= 2025) & (d.peril == "haze")].paid.sum() / annual.sum() * 100})
pricing = pd.DataFrame(rows).set_index(["segment", "area"]).round(2)
assert set(pricing.index.names) == {"segment", "area"}, "price by segment, area and peril only (FEAT)"
pricing

### Who pays, and is it affordable?
Korea's stallholder cover failed because nobody would carry the premium. So we decide the **split before launch**. Any start-up subsidy is shown **on the premium line with an end date**; there is no permanent subsidy.

Affordability checks (on the customer's share **after** the subsidy ends, so we don't rely on it):
- **A few dollars a week** for the customer's own share.
- The **Philippines rule**: a micro-premium should cost at most **7.5% of a day's minimum wage per day**. Singapore has no national minimum wage, so we use a reference lower-wage daily income of S$80 (an assumption: about S$2,080 over 26 working days).
- For reference: Malaysia's Perlindungan Tenang costs RM50–75 a year, and the Asia-Pacific average microinsurance premium is about US$13 a year.

In [ ]:
SUBSIDY_END = "31 Dec 2027"
PREMIUM_SPLIT = {   # who pays what share of the gross premium (fictional proposal, to negotiate)
    "Outdoor / platform worker": {"Worker": 0.4, "Platform / employer": 0.4, f"Start-up subsidy (ends {SUBSIDY_END})": 0.2},
    "Hawker / small shop": {"Stallholder": 0.6, "Hawker association / landlord": 0.2, f"Start-up subsidy (ends {SUBSIDY_END})": 0.2},
    "Renter household": {"Household": 0.5, "Union / community partner": 0.3, f"Start-up subsidy (ends {SUBSIDY_END})": 0.2},
}
REF_DAILY_WAGE = 80

def premium_split(segment, area):
    weekly = pricing.loc[(segment, area), "per_week"]
    split = PREMIUM_SPLIT[segment]
    subsidy = sum(v for k, v in split.items() if "subsidy" in k)
    after = {k: 0 if "subsidy" in k else v / (1 - subsidy) for k, v in split.items()}  # others take over the subsidy share
    return pd.DataFrame({"share_now": split,
                         "S$_per_week_now": {k: weekly * v for k, v in split.items()},
                         "S$_per_week_from_2028": {k: weekly * v for k, v in after.items()}}).round(2)

afford = []
for (seg, area), r in pricing.iterrows():
    split = premium_split(seg, area)
    own = split.iloc[0]["S$_per_week_from_2028"]                # customer (listed first), after the subsidy ends
    afford.append({"segment": seg, "area": area, "customer_S$_per_week_now": split.iloc[0]["S$_per_week_now"],
                   "customer_S$_per_week_from_2028": own, "customer_S$_per_year_from_2028": own * 52,
                   "pct_of_daily_wage_per_day": own / 7 / REF_DAILY_WAGE * 100,
                   "few_dollars_a_week": own <= 5, "within_7.5pct_rule": own / 7 <= 0.075 * REF_DAILY_WAGE})
affordability = pd.DataFrame(afford).set_index(["segment", "area"]).round(2)
display(premium_split("Outdoor / platform worker", "Jurong West"))
affordability

## Step 7: The One-SMS Explainer (Batch Structured Output)
Income's rule: trigger, payout and exclusions must **fit in one SMS in the customer's language**, including one plain sentence that says **when it does not pay**.

**Coaching point on cost:** 3 segments × 4 languages = 12 messages, written in **one** request. Python builds the fact sheet from the config, so the LLM only *translates and simplifies* facts; it never invents them.

Then **Python checks** each SMS:
- **Length:** English fits the GSM-7 alphabet (160 characters, or 153 per part when joined). Chinese and Tamil need **UCS-2**, which allows only **70 characters** (67 per part). A long message arrives as one SMS on the phone but is billed per part, so we allow up to 3 parts.
- **Numbers:** every number in the SMS must appear in the config. A wrong payout amount in a sales message is a fair-dealing problem.

In [ ]:
import re
from typing import Literal
from pydantic import BaseModel

LANGUAGES = ["English", "Chinese", "Malay", "Tamil"]

def steps(pays):
    """Keep only tiers where the payout goes up, e.g. drop 'Hazardous' if it pays the same as 'Very unhealthy'."""
    out = {}
    for tier, amount in pays.items():
        if amount > max(out.values(), default=0):
            out[tier] = amount
    return out

def fact_sheet(segment):
    s = SEGMENTS[segment]
    lines = []
    if s["pays"]["heat"]:
        lines.append(f"Heat: S${s['pays']['heat']['High']} when the NEA station heat stress (WBGT) reaches "
                     f"{HEAT_BANDS['High'] - s['wbgt_adj']:g} or more (we add {s['wbgt_adj']:g} for worksite heat, so it counts as High)")
    rain, haze = steps(s["pays"]["rain"]), steps(s["pays"]["haze"])
    lines.append("Rain at your area gauge: " + ", ".join(f"S${v} if {RAIN_BANDS[k]}mm or more in a day" for k, v in rain.items()))
    lines.append("Haze, 24-hour PSI: " + ", ".join(f"S${v} if PSI {HAZE_BANDS[k]} or more" for k, v in haze.items()))
    lines.append(f"Max {s['cap_days']} paid days a year, one payout per day. Paid to PayNow automatically, no claim form.")
    lines.append("Does NOT pay: if the reading stays below these levels you get nothing, even if you lose money. No injury or medical cover. Starts 14 days after joining.")
    return f"WeatherShield by Income, for {segment}:\n" + "\n".join(f"- {l}" for l in lines)

def allowed_numbers(segment):
    s = SEGMENTS[segment]
    nums = {s["cap_days"], 14, 24, s["wbgt_adj"], HEAT_BANDS["High"] - s["wbgt_adj"]}
    nums |= set(RAIN_BANDS.values()) | set(HAZE_BANDS.values()) | {300}
    nums |= {v for peril in s["pays"].values() for v in peril.values()}
    return {float(x) for x in nums}

class SmsExplainer(BaseModel):
    segment: Literal[tuple(SEGMENTS)]
    language: Literal[tuple(LANGUAGES)]
    sms: str

facts = "\n\n".join(fact_sheet(s) for s in SEGMENTS)
SMS_PROMPT = f"""You write SMS product explainers for Income Insurance, a licensed insurer in Singapore.
Write ONE SMS for EVERY segment in EVERY language: {", ".join(LANGUAGES)} (Chinese = Simplified Chinese).
Each SMS must state: what triggers a payout, how much, the yearly cap, and this idea in one plain sentence:
"It pays only on the official readings, so you can lose income and get nothing."
Rules: use ONLY the facts below, use Western digits (0-9) and "S$", no other numbers, no emojis, no links.
Keep it short: English under 300 characters; Chinese under 120 characters; Malay under 300; Tamil under 200.

FACTS:
{facts}"""

sms_list = ask_gemini(SMS_PROMPT, types.GenerateContentConfig(
    response_mime_type="application/json", response_schema=list[SmsExplainer])).parsed
SMS = {(m.segment, m.language): m.sms for m in sms_list}

GSM7 = set("@£$¥èéùìòÇ\nØø\rÅåΔ_ΦΓΛΩΠΨΣΘΞÆæßÉ !\"#¤%&'()*+,-./0123456789:;<=>?¡ABCDEFGHIJKLMNOPQRSTUVWXYZÄÖÑÜ§¿abcdefghijklmnopqrstuvwxyzäöñüà")

def sms_check(segment, text):
    gsm = set(text) <= GSM7
    single, part = (160, 153) if gsm else (70, 67)
    parts = 1 if len(text) <= single else -(-len(text) // part)
    found = {float(x) for x in re.findall(r"\d+(?:\.\d+)?", text)}
    must = {float(v) for peril in SEGMENTS[segment]["pays"].values() for v in peril.values()}
    return {"encoding": "GSM-7" if gsm else "UCS-2", "chars": len(text), "sms_parts": parts, "fits_3_parts": parts <= 3,
            "unknown_numbers": sorted(found - allowed_numbers(segment)), "missing_payouts": sorted(must - found)}

sms_checks = pd.DataFrame([{"segment": seg, "language": lang, **sms_check(seg, text)} for (seg, lang), text in SMS.items()])
print(f"✅ {len(SMS)} SMS written in 1 API call\n")
for lang in LANGUAGES:
    print(f"[{lang}] {SMS.get(('Outdoor / platform worker', lang), '(missing)')}\n")
sms_checks

## Step 8: The Say-Back Test (LLM-as-Judge)
Income's must-have: the customer can **say back** what they pay for and when they get paid, **in under a minute**. Real teams should test with real people. As a quick, cheap first check we use two LLM calls:
1. **Simulated customers** read *only* the SMS and paraphrase it in their own words, the way a busy rider or auntie would.
2. A separate **judge** compares each paraphrase with the true fact sheet and returns a structured verdict: did they get the **trigger**, the **payout**, and the **no-pay case**?

Keeping the judge separate from the writer avoids the model marking its own homework.

In [ ]:
PERSONAS = {
    "English": "a 24-year-old food-delivery rider, reads quickly between orders",
    "Chinese": "a 61-year-old auntie who helps at a drinks stall, reads Chinese only",
    "Malay": "a 45-year-old private-hire driver, reads Malay best",
    "Tamil": "a 38-year-old cleaner who works outdoors, reads Tamil best",
}
SEG = "Outdoor / platform worker"

class SayBack(BaseModel):
    language: Literal[tuple(LANGUAGES)]
    paraphrase: str

class Verdict(BaseModel):
    language: Literal[tuple(LANGUAGES)]
    understands_trigger: bool
    understands_payout: bool
    understands_no_pay_case: bool
    misunderstanding: str
    suggested_fix: str

sms_block = "\n\n".join(f"[{lang}] customer: {PERSONAS[lang]}\nSMS: {SMS.get((SEG, lang), '')}" for lang in LANGUAGES)
saybacks = ask_gemini(
    "Role-play each customer below. They read ONLY this SMS once. In 2-3 simple English sentences, say back what "
    "they think the product does: when it pays, how much, and when it doesn't. Do not add facts that are not in "
    "the SMS; if something is unclear, write what this person would probably guess.\n\n" + sms_block,
    types.GenerateContentConfig(response_mime_type="application/json", response_schema=list[SayBack]),
).parsed

said = "\n".join(f"[{s.language}] {s.paraphrase}" for s in saybacks)
verdicts = ask_gemini(
    f"You are a strict compliance reviewer checking customer understanding.\nTRUE PRODUCT FACTS:\n{fact_sheet(SEG)}\n\n"
    f"CUSTOMER SAY-BACKS:\n{said}\n\nFor each language, judge whether the customer correctly understood the trigger, "
    "the payout amounts, and that it can pay nothing even when they lose income. Name any misunderstanding and a short fix to the SMS.",
    types.GenerateContentConfig(response_mime_type="application/json", response_schema=list[Verdict]),
).parsed

say_back = pd.DataFrame([v.model_dump() for v in verdicts])
say_back["pass"] = say_back[["understands_trigger", "understands_payout", "understands_no_pay_case"]].all(axis=1)
for s in saybacks:
    print(f"🗣️ [{s.language}] {s.paraphrase}")
say_back

## Step 9: Point of Sale: Insurable Interest Check
Income must sell only to people who would **really lose money**, and size the payout to the **expected loss**. Otherwise it's a bet, not insurance (ZhongAn's lesson).

Applicants describe themselves in free text. The **LLM only extracts** a few fields into a schema. **Python rules decide** eligibility and cover size, so every decision is explainable and auditable.

**PDPA:** we collect only the fields needed for this decision. The schema has **no field** for names, NRIC, phone numbers or health, so even if the applicant types them, they are not kept.

**Cover size:** each segment has three plans (Lite ×0.5, Standard ×1, Plus ×1.5 of the payouts in Step 4). We offer the **largest plan whose biggest daily payout stays at or below the applicant's expected loss** on a bad-weather day.

In [ ]:
WORK_TYPES = ["outdoor or platform worker", "hawker or shop owner", "renter household", "indoor or other"]
TO_SEGMENT = dict(zip(WORK_TYPES[:3], SEGMENTS))
PLANS = {"Plus": 1.5, "Standard": 1.0, "Lite": 0.5}

class Applicant(BaseModel):
    applicant_id: int
    work_type: Literal[tuple(WORK_TYPES)]
    outdoor_hours_per_day: float
    income_or_sales_per_day_sgd: float
    area: Literal[tuple(list(AREAS) + ["Other or unknown"])]
    rents_home: bool
    evidence: str

APPLICANTS = [   # fictional applicants
    "Hi, I deliver food on my e-bike about 6 hours a day, mostly around Jurong West. Normal day I make about $110 after charging the bike.",
    "I'm an accountant in Raffles Place, aircon office all day. My IC is S0000000X. Heard it's cheap so I want to buy for the payouts lah.",
    "Auntie here, I sell chicken rice at a hawker centre in Kallang, 10am to 8pm. Sales about $600 a day, but heavy rain nobody comes.",
    "We rent a 2-room flat on the ground floor in Bedok. I'm retired. Water came into the flat during the big storm last year.",
]

def extract(texts):
    numbered = "\n".join(f"Applicant {i}: {t}" for i, t in enumerate(texts, 1))
    return ask_gemini(
        "Extract insurance application fields for each applicant. Use 0 when hours or income are not stated. "
        "Use 'Other or unknown' if the area is not one of the listed areas. 'evidence' = a short quote showing their work. "
        "Do NOT copy names, IC/NRIC numbers, phone numbers or health details.\n\n" + numbered,
        types.GenerateContentConfig(response_mime_type="application/json", response_schema=list[Applicant]),
    ).parsed

def decide(a):
    """Python rules: insurable interest, then the largest plan within the expected daily loss."""
    seg = TO_SEGMENT.get(a.work_type)
    if seg is None:
        return {"eligible": False, "segment": "-", "plan": "-", "expected_loss_per_bad_day": 0,
                "reason": "No insurable interest: weather does not stop this work, so a payout would be a bet, not cover."}
    if seg == "Outdoor / platform worker":
        loss, ok = a.income_or_sales_per_day_sgd * min(a.outdoor_hours_per_day, 8) / 8 * 0.5, a.outdoor_hours_per_day >= 3
        why = "needs at least 3 outdoor working hours a day"
    elif seg == "Hawker / small shop":
        loss, ok, why = 0.3 * a.income_or_sales_per_day_sgd, a.income_or_sales_per_day_sgd > 0, "needs daily sales to size cover"
    else:
        loss, ok, why = 500, a.rents_home and a.area in FLOOD_PRONE, f"needs a rented home in a flood-prone area ({', '.join(FLOOD_PRONE)})"
    if a.area not in SEGMENTS[seg]["areas"]:
        ok, why = False, "area not covered yet"
    top = max(v for peril in SEGMENTS[seg]["pays"].values() for v in peril.values())
    plan = next((p for p, x in PLANS.items() if top * x <= loss), None)
    if not ok or plan is None:
        return {"eligible": False, "segment": seg, "plan": "-", "expected_loss_per_bad_day": round(loss, 2),
                "reason": f"Not eligible: {why}." if not ok else "Even the Lite plan would pay more than the expected loss."}
    return {"eligible": True, "segment": seg, "plan": plan, "expected_loss_per_bad_day": round(loss, 2),
            "reason": f"Biggest daily payout S${top * PLANS[plan]:g} is within the expected loss of S${loss:.0f} on a bad-weather day."}

applicants = extract(APPLICANTS)
decisions = pd.DataFrame([{**a.model_dump(exclude={"evidence"}), **decide(a)} for a in applicants])
decisions

## Step 10: Write the Proposal Section (pandas calculates, LLM narrates)
LLMs are **bad at arithmetic** on long tables and will confidently give a wrong total. So pandas computed every number above. Now we hand the LLM a small, accurate summary and ask it to write the **back-test and pricing** part of the proposal for Income.

In [ ]:
seg_view = backtest_summary.groupby(level="segment").mean().round(1)
price_view = pricing.groupby(level="segment")[["pure_premium", "gross_per_year", "per_week", "haze_share_pct"]].agg(["min", "max"]).round(2)
summary = f"""PRODUCT: WeatherShield, parametric heat/rain/haze income cover designed with Income (licensed insurer).
BACK-TEST 2015-2025, average over areas, per segment:
{seg_view.to_string()}
HEAT TRIGGER SET AT RECORD WBGT {RECORD_WBGT} (SEWA 2023 MISTAKE): {record_zero_years} of 11 years with zero heat payouts; our High-band heat trigger: {(heat_check.iloc[:, 0] == 0).sum()} zero years
PRICING (S$, min and max across areas):
{price_view.to_string()}
EXPENSE RATIO: {EXPENSE_RATIO:.0%}; RISK LOADING: {RISK_LOAD:.0%} of payout std dev
CUSTOMER SHARE PER WEEK, MAX ACROSS AREAS (now / after subsidy ends {SUBSIDY_END}): {affordability['customer_S$_per_week_now'].groupby(level='segment').max().round(2).to_dict()} / {affordability['customer_S$_per_week_from_2028'].groupby(level='segment').max().round(2).to_dict()}
SAY-BACK TEST PASS RATE (outdoor worker SMS): {say_back['pass'].mean():.0%}"""
print(summary)

PROPOSAL_PROMPT = f"""You are helping a student team write a product proposal for Income Insurance in Singapore.
Using ONLY the numbers in this summary (never calculate new ones), write the "Back-test and pricing" section:
3 short paragraphs (max 200 words in total) covering how often it pays and the zero-payout lesson, basis risk
(loss days with no payout) and what the team would do about it, and price and affordability including who pays.
Plain English, no hype.

SUMMARY:
{summary}"""
proposal_text = ask_gemini(PROPOSAL_PROMPT).text
print("\n📝 Draft proposal section:\n", proposal_text)

## Step 11: Build the Demo UI with Gradio 🏆
Two tabs:
- **📦 Product & back-test:** pick a segment, area and language. Everything shown was computed above, so it's instant and makes **no LLM calls**.
- **✅ Check my eligibility:** an applicant types a sentence; **1 LLM call** per click extracts the fields, then the Python rules decide.

In [ ]:
import gradio as gr

def product_view(segment, area, language):
    if area not in SEGMENTS[segment]["areas"]:
        msg = f"⚠️ {segment} cover is only offered in flood-prone areas: {', '.join(SEGMENTS[segment]['areas'])}."
        return tier_table(segment), None, msg, pd.DataFrame(), "", pd.DataFrame()
    p = pricing.loc[(segment, area)]
    split = premium_split(segment, area).reset_index(names="payer")
    premium_md = (f"### S${p.per_week:.2f} a week (S${p.gross_per_year:.0f} a year)\n"
                  f"Pure premium S${p.pure_premium:.0f} + risk loading S${p.risk_loading:.0f}, ÷ (1 − {EXPENSE_RATIO:.0%} expenses). "
                  f"Up to {SEGMENTS[segment]['cap_days']} paid days a year. Subsidy shown below ends **{SUBSIDY_END}**.")
    history = yearly(results[(segment, area)]).reset_index()
    sms = SMS.get((segment, language), "(SMS not generated)")
    chk = sms_check(segment, sms)
    sms_md = f"{sms}\n\n*{chk['encoding']}, {chk['chars']} characters, {chk['sms_parts']} SMS part(s)*"
    return tier_table(segment), backtest_chart(segment, area), premium_md, split, sms_md, history

def check_eligibility(text):
    if not text.strip():
        return "Please describe your work and where you live or work."
    try:
        a = extract([text])[0]
    except Exception as e:
        return f"⚠️ Sorry, something went wrong: {e}"
    r = decide(a)
    head = f"## {'✅ Eligible: ' + r['segment'] + ', ' + r['plan'] + ' plan' if r['eligible'] else '❌ Not eligible'}"
    return (f"{head}\n{r['reason']}\n\n**What we recorded (only these fields):** work type `{a.work_type}`, "
            f"outdoor hours {a.outdoor_hours_per_day:g}/day, income or sales S${a.income_or_sales_per_day_sgd:g}/day, "
            f"area `{a.area}`, renter: {a.rents_home}.")

with gr.Blocks(title="WeatherShield") as demo:
    gr.Markdown("# 🌦️ WeatherShield by Income (prototype)\nParametric climate income cover for heat, rain and haze. "
                "*Synthetic data, for demonstration only.*")
    with gr.Tab("📦 Product & back-test"):
        with gr.Row():
            seg_in = gr.Dropdown(list(SEGMENTS), value="Outdoor / platform worker", label="Segment")
            area_in = gr.Dropdown(list(AREAS), value="Jurong West", label="Area")
            lang_in = gr.Dropdown(LANGUAGES, value="English", label="SMS language")
        with gr.Row():
            tiers_out = gr.Dataframe(label="Triggers and payout per day (S$0 rows = no payout)")
            with gr.Column():
                premium_out = gr.Markdown()
                split_out = gr.Dataframe(label="Who pays (per week)")
        chart_out = gr.Plot(label="Back-test 2015-2026")
        sms_out = gr.Markdown(label="One-SMS explainer")
        history_out = gr.Dataframe(label="Payout history per year")
        outputs = [tiers_out, chart_out, premium_out, split_out, sms_out, history_out]
        for c in (seg_in, area_in, lang_in):
            c.change(product_view, [seg_in, area_in, lang_in], outputs)
        demo.load(product_view, [seg_in, area_in, lang_in], outputs)
    with gr.Tab("✅ Check my eligibility"):
        text_in = gr.Textbox(lines=3, label="Tell us about your work (no names or IC numbers needed)",
                             placeholder="e.g. I drive private-hire about 8 hours a day around Woodlands, about $150 a day.")
        go = gr.Button("Check", variant="primary")
        result_out = gr.Markdown()
        gr.Examples(APPLICANTS, inputs=text_in)
        go.click(check_eligibility, text_in, result_out)

# Launch it! (This will generate a public URL you can share with judges)
demo.launch(share=True, theme="soft")

## 🎯 What the Sponsor Wants After 36 Hours
Income's strong outcome is a **product proposal** covering these 8 items, each with a justification:
- [ ] **Concept / design** of the product
- [ ] **Target group(s)**
- [ ] **Simple pricing logic and unit economics**
- [ ] **Clear claim parameters and exclusions**
- [ ] **Expected impact** on financial protection, and how affordable it is
- [ ] **Market competitiveness**
- [ ] **Distribution proposal**
- [ ] **How AI and data are integrated**

Copy this template into your proposal and fill it in:

| # | Proposal item | Your answer | Justification / evidence | Where this notebook helps |
|---|---|---|---|---|
| 1 | Concept / design | | | Step 4 config, tier table |
| 2 | Target group(s) | | | `SEGMENTS[...]["who"]`, Step 9 eligibility |
| 3 | Pricing logic & unit economics | | | Step 6 `pricing`, `premium_split`, `affordability` |
| 4 | Claim parameters & exclusions | | | `tier_table`, `EXCLUSIONS`, SMS |
| 5 | Expected impact & affordability | | | `backtest_summary` (loss days paid), affordability checks |
| 6 | Market competitiveness | | | Compare with Droplet, Singlife rain cover, Swiss Re haze cover, SmartBiz 360 |
| 7 | Distribution | | | `PREMIUM_SPLIT` (platform, union, stall associations), SMS in 4 languages |
| 8 | AI & data integration | | | Public NEA/PUB/MSS data, Steps 7–10 |

## ✅ Constraints Checklist
- [x] **Offered by/with Income as licensed insurer:** branded "WeatherShield by Income"; we design the product, Income underwrites it.
- [x] **Independent, verifiable triggers:** NEA WBGT, 24-hour PSI and rain gauges, published on data.gov.sg (prototype uses synthetic history plus an optional live peek).
- [x] **Official bands people already act on:** NEA heat stress Low / Moderate / High, PSI Unhealthy / Very unhealthy.
- [x] **Worksite vs station gap handled:** fixed, published `wbgt_adj` per segment. ⚠️ Validate with worksite readings.
- [x] **Say plainly when it does not pay:** S$0 tier rows, `EXCLUSIONS`, and the no-pay sentence in every SMS.
- [x] **Back-tested:** payouts per year, zero-payout years and basis risk. ⚠️ On synthetic data; redo with real NEA/PUB/MSS history.
- [x] **Insurable interest at point of sale:** Step 9 rules, cover capped by expected daily loss.
- [x] **Who pays decided up front:** `PREMIUM_SPLIT`, subsidy with an end date on the premium line.
- [x] **Cap frequency and pool perils:** yearly cap on paid days; heat, rain and haze bundled; risk loading for correlated haze. ⚠️ Reinsurance cost is only approximated.
- [x] **Affordability:** a few dollars a week and the 7.5% rule checked; expenses at 20%.
- [x] **Understanding in one SMS, in 4 languages:** length and number checks, plus a say-back test. ⚠️ Simulated customers are no substitute for real ones.
- [x] **PDPA and FEAT:** minimal schema, no demographics in pricing, rules not black boxes.
- [x] **Complements statutory cover:** excludes injury and medical costs.
- [x] **No new hardware:** public stations only.
- [ ] **Pay based on needs:** we pay the next day by PayNow; timing payouts to the *start* of a forecast heatwave (Delhi trial) is an extension.
- [ ] **PUB flood sensors:** not used yet; the renter trigger uses rainfall only.

## 🚀 Make It Your Own: Hackathon Extensions
Pick one or two. Don't try all of them.
- **Real back-test:** download the data.gov.sg WBGT, rainfall and PSI history and MSS daily records, and replace the synthetic `wx` table.
- **Hourly triggers:** pay when WBGT is High for 3+ hours in working time, or rain ≥ 30 mm in 2 hours. Re-run the basis-risk numbers.
- **PUB flood sensors:** trigger the renter cover on a nearby water-level sensor instead of daily rain.
- **Pay before the loss:** pay a small amount on the first day of an NEA heatwave *forecast*, like the Delhi trial.
- **SME bundle:** extend Income's SmartBiz 360 with a climate add-on for lost trade. Price it with this notebook.
- **Risk reduction rewards:** cooling vests or flood barriers bundled with cover, or a premium discount for taking MOM rest breaks.

## 🎤 Pitching to Judges
- Lead with **one customer and one number**: *"A Jurong West rider pays about S$X a week and would have been paid in every one of the last 11 years."*
- Show the **back-test chart** and be honest about **basis risk**. Judges trust a team that shows where its product fails.
- Show the **SMS** in Chinese or Tamil and the **say-back** score. That's Income's "understand in under a minute" test.
- Explain **who pays** and **when the subsidy ends**.
- Mention cost: **5 LLM calls** for the whole pipeline; pandas does the maths.

## 🛡️ Responsible AI Checklist
- [ ] Synthetic data only. **Never put real customer data into the free Gemini tier**, which may use inputs to improve Google's products.
- [ ] **Explainable pricing (MAS FEAT):** inputs are area and peril only; the formula fits on one line.
- [ ] **Rules decide, AI extracts:** eligibility and payouts come from Python rules you can audit, not from the LLM.
- [ ] **PDPA:** collect only the fields you need, say why, get consent, and don't store free text longer than necessary.
- [ ] **Fair dealing:** every number in customer messages is checked against the config; a human reviews translations before launch.
- [ ] **Test with real people:** run the say-back test with real riders, hawkers and seniors in each language.